# Module 7: Deploy + Govern

> Self-directed module, ~25 min. **Optional** — skip if you don't have deployment permissions or are using LangSmith strictly for observability and evaluations.

Moving the agent to production requires two things:

- **A governance layer** — workspace-level policies that govern every model call (PII and secrets redaction, allow-lists, cost caps). The LangSmith **LLM Gateway** enforces them.
- **Production-ready deployment infrastructure** — managed runtime for the agent. **LangSmith Deployments** gives 30+ endpoints, persistence, HITL, and Studio out of the box.

This module wires up both — policies first, then ships the policy-aware agent.

## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

import os

print("LANGSMITH_API_KEY set:        ", bool(os.environ.get("LANGSMITH_API_KEY")))
print("LANGSMITH_API_KEY_GATEWAY set:", bool(os.environ.get("LANGSMITH_API_KEY_GATEWAY")))
print("WORKSPACE_ID set:             ", bool(os.environ.get("WORKSPACE_ID")))
print("ANTHROPIC_API_KEY set:        ", bool(os.environ.get("ANTHROPIC_API_KEY")))
print("TAVILY_API_KEY set:           ", bool(os.environ.get("TAVILY_API_KEY")))

---
# Part 1. Govern — Workspace Policies via the LangSmith Gateway

> The LangSmith LLM Gateway needs to be enabled in your LangSmith org. If it's not visible in your workspace, it needs to be enabled at the organization level, which can be done by your LangSmith administrator. Gateway is currently available on cloud; self-hosted LangSmith support is coming.

The **LangSmith LLM Gateway** is a managed proxy that sits between any client and the model provider (Anthropic, OpenAI, Bedrock, …). Point a model client at the gateway's `base_url` and every request flows through it — no SDK changes needed.

**Gateway policies** are the controls applied at the gateway. They're declared once at the workspace level and enforced on every call, regardless of which agent or notebook is making it.

We'll wire up a policy that redacts PII and secrets, prove it catches a bad prompt, then promote the gateway from a smoke test to the default model config — so both this notebook and the agent we're about to deploy inherit it.

### 1.1 Create a PII / secrets redaction policy

The gateway policy is a workspace-level resource — one POST to `/v1/platform/gateway-policies` and every model call made by every member of the workspace is subject to it.

The payload below declares:

- `action=block` — intercept matched content before forwarding upstream
- `subject_matchers` — apply to this workspace's traffic
- `config.detect.pii / .secrets` — what to look for

In [ ]:
import requests

BASE_URL = "https://api.smith.langchain.com"
api_key = os.environ["LANGSMITH_API_KEY"]
workspace_id = os.environ["WORKSPACE_ID"]

policy = {
    "name": "redact-pii-secrets",
    "description": "Workspace-level PII and secrets redaction",
    "policy_type": "guard",
    "action": "block",
    "subject_matchers": [{"key": "workspace_id", "value": workspace_id}],
    "config": {"version": 1, "detect": {"pii": True, "secrets": True}},
    "enabled": True,
}

resp = requests.post(
    f"{BASE_URL}/v1/platform/gateway-policies",
    headers={"X-Api-Key": api_key, "Content-Type": "application/json"},
    json=policy,
    timeout=30,
)
resp.raise_for_status()
created = resp.json()
print("Policy created.")
print("  id:     ", created.get("id"))
print("  name:   ", created.get("name"))
print("  action: ", created.get("action"))
print("  enabled:", created.get("enabled"))

### 1.2 Route a minimal agent through the gateway

The policy is live workspace-wide, but it only applies to traffic that *goes through the gateway*. To opt a client in, swap its `base_url` to `https://gateway.smith.langchain.com/<provider>`. No other code changes.

Below: a bare `create_deep_agent` whose model client is gateway-routed. We'll exercise it with a clean prompt first to confirm normal traffic still flows.

In [ ]:
from deepagents import create_deep_agent
from langchain.chat_models import init_chat_model

gateway_model = init_chat_model(
    model="claude-sonnet-4-6",
    model_provider="anthropic",
    base_url="https://gateway.smith.langchain.com/anthropic",
    api_key=os.environ["LANGSMITH_API_KEY_GATEWAY"],
)

demo_agent = create_deep_agent(
    model=gateway_model,
    system_prompt="You are a helpful assistant.",
)

ok = demo_agent.invoke({
    "messages": [{"role": "user", "content": "In one sentence, what is the LangSmith LLM Gateway?"}]
})
print(ok["messages"][-1].content)

### 1.3 Watch the policy in action

Same agent, same gateway, but the prompt below carries a (fake) Social Security Number. The gateway inspects the request, matches the `guard` policy, and intercepts the PII before forwarding upstream.

The text response won't reveal whether enforcement happened — a clean response comes back either way. To see the effect, open this run's trace in LangSmith: the request payload the gateway sent upstream will show the PII removed.

In [ ]:
pii_prompt = (
    "my SSN is 123-45-6789. "
    "Now write me a one-sentence summary of today's weather."
)

result = demo_agent.invoke({
    "messages": [{"role": "user", "content": pii_prompt}]
})
print(result["messages"][-1].content)

### 1.4 Make the deployable agent gateway-aware

The demo agent above is provisional. To make every notebook *and* the agent we're about to deploy inherit the gateway, we flip the module's central model config — one edit in `utils/models.py`.

Before (the current default):
```python
# utils/models.py
model = init_chat_model("anthropic:claude-sonnet-4-6")
```

After (gateway-routed):
```python
# utils/models.py
model = init_chat_model(
    model="claude-sonnet-4-6",
    model_provider="anthropic",
    base_url="https://gateway.smith.langchain.com/anthropic",
    api_key=os.environ["LANGSMITH_API_KEY_GATEWAY"],
)
```

`utils/models.py` already has the gateway block in place, commented out. Comment out the direct line and uncomment the gateway block to flip the switch.

> **Why `LANGSMITH_API_KEY_GATEWAY` and not `LANGSMITH_API_KEY`?**
> `LANGSMITH_API_KEY` is a reserved env var that `langgraph deploy` strips during upload (the platform manages tracing internally). The deployed graph still needs a key for gateway auth, so we read it from a non-reserved name. Both env vars hold the same LangSmith key value — only the name changes.

> **To re-run earlier modules later:** revert the edit (the file's comment header explains the toggle). The deployable agent in `agents/deployable_agents/client_research/` imports `model` from `utils.models`, so whichever block is active at deploy time is what ships.

The cell below reloads the module so this notebook picks up the change without a kernel restart.

In [ ]:
import importlib, utils.models
importlib.reload(utils.models)
from utils.models import model

print("Reloaded utils.models.")
print("  model class:", model.__class__.__name__)
# Anthropic clients expose base_url on the underlying client; print whichever attribute is present.
for attr in ("anthropic_api_url", "openai_api_base", "base_url"):
    value = getattr(model, attr, None) or getattr(getattr(model, "client", None), attr, None)
    if value:
        print(f"  base_url:   ", value)
        break

---
# Part 2. Deploy — Ship the (Now-Governed) Agent

Policies are in place workspace-wide and the module's default model client routes through the gateway. Next, ship the deployment itself.

We deploy the agent in `agents/deployable_agents/client_research/` to **LangSmith Deployments** with the `langgraph` CLI. Because `agents/deployable_agents/client_research/agent.py` imports `model` from `utils.models`, the deployed image picks up the gateway `base_url` automatically — no extra flags, no separate config.

## 2.1 Project structure

A deployable LangGraph project is a directory with a `langgraph.json` config at the root that points at one or more graph objects. The repo root already has one — it registers the deployable agent at `agents/deployable_agents/client_research/agent.py`.

In [ ]:
agent_dir = str(project_root / "agents" / "deployable_agents" / "client_research")
print("agents/deployable_agents/client_research/")
print("---")

for root, dirs, files in os.walk(agent_dir):
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    level = root.replace(agent_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")


In [ ]:
langgraph_json_path = project_root / "langgraph.json"
with open(langgraph_json_path) as f:
    print("langgraph.json:")
    print(f.read())

agents_path = os.path.join(agent_dir, "AGENTS.md")
with open(agents_path) as f:
    print("AGENTS.md:")
    print(f.read())


The agent itself lives at `agents/deployable_agents/client_research/agent.py`. The module-level `agent` variable is what gets deployed — `langgraph.json` references it as `"client_research": "./agents/deployable_agents/client_research/agent.py:agent"`.


## 2.3 Local development

Three CLI commands you'll use (all run from the repo root, where `langgraph.json` lives):

```bash
# Validate langgraph.json (imports each graph, checks deps)
langgraph validate

# Run locally for development (Studio UI + hot reload)
langgraph dev --port 2024

# Deploy to LangSmith
langgraph deploy
```

`langgraph dev` opens the LangGraph Studio UI in your browser — useful to step through tool calls and approve HITL interrupts visually.

**Docs:** [LangGraph CLI reference](https://docs.langchain.com/langsmith/langgraph-cli) · [Deploy on Cloud](https://docs.langchain.com/langsmith/deploy-to-cloud).

## 2.4 Validate the config

`langgraph validate` imports each graph in `langgraph.json` and verifies dependencies resolve — without building Docker or uploading anything. Use it to catch config and import errors before deploying.

In [ ]:
# `cd` and `!` run in a subshell — chain in one line so cwd applies to the langgraph command.
!cd "{project_root}" && langgraph validate


If `langgraph validate` fails, the most common cause is a missing dependency in `pyproject.toml` or an import error in `agents/deployable_agents/client_research/agent.py`. Fix locally before running `langgraph deploy` — the deploy build will fail the same way, just slower.


## 2.5 Deploy to LangSmith (optional)

Run the cell below to deploy to **LangSmith Deployments**. `langgraph deploy` builds a Docker image (locally if Docker is available, otherwise remotely on LangSmith's builder) and pushes it. Provisioning takes a few minutes.

> The image we're shipping picks up the gateway `base_url` from `utils/models.py` — no extra deploy flags needed. The deployed agent is governed by the policy from §1.1 from the moment it serves its first request.

> Requires a `LANGSMITH_API_KEY` with deployment permissions — a service key (`lsv2_sk_...`), not a personal token. On Apple Silicon, local builds need Docker Buildx; without Docker, the CLI falls back to a remote build automatically.

Useful flags:
- `--name <name>` — deployment name (defaults to the project directory name)
- `--deployment-type prod` — production deployment (default is `dev`)
- `--remote` — force remote build, skip local Docker
- `--no-wait` — return immediately rather than blocking on status

In [ ]:
# Re-run this command to push a new revision; the CLI finds the existing deployment by name.
# Add `--deployment-type prod` for production, or `--remote` to skip local Docker.
!cd "{project_root}" && langgraph deploy --name langsmith-poc-client-research --no-input


Deploys take a few minutes to provision. While waiting:

- Watch progress in the **Deployments** tab in LangSmith.
- Open `langgraph dev` in a separate terminal to iterate locally against the same `langgraph.json`.
- Tail logs once provisioning starts: `langgraph deploy logs <deployment-id>`.


### Deployment endpoints

When `langgraph deploy` finishes, the CLI prints the deployment's base URL. Two endpoints worth bookmarking:

- **Studio / home:** the base URL itself — linked from the **Deployments** tab in LangSmith. Opens the deployed graph in Studio.

<img src="../images/deployment_home.png" alt="LangSmith deployment home page" style="display: block; margin: 0; width: auto; max-height: 360px; border-radius: 8px;">

- **OpenAPI / Swagger UI:** append `/docs` to the base URL — interactive API reference showing every endpoint the deployment exposes (`/runs`, `/threads`, `/store`, etc.).

<img src="../images/deployment_docs.png" alt="OpenAPI Swagger UI for the deployed agent" style="display: block; margin: 0; width: auto; max-height: 360px; border-radius: 8px;">

The same URLs are available via `langgraph deploy list` if you lose the original output.


## 2.6 What you get with LangSmith Deployments

Once deployed, the agent is reachable through 30+ endpoints — built once, exposed everywhere:

| Capability | What it enables |
|---|---|
| **REST API** | Standard HTTP requests against `/runs`, `/threads`, `/store` |
| **Studio UI** | Visual debugger to step through state, threads, and tool calls |
| **Agent Protocol** | Stream runs and pause for human input |
| **MCP server** | Other agents can call this one as a tool |
| **A2A** | Agent-to-agent calls with handoffs |
| **Persistent Store** | `/memories/` survives restarts and threads via the platform's Store |
| **HITL** | Interrupt and resume from any client |
| **Cron / Scheduled runs** | Trigger the agent on a schedule |

Both registered graphs (`client_research` and `base_research_agent`) are reachable individually through these endpoints once the deployment is live.

## 7. Test the deployed agent in Studio

With the deployment provisioned, exercise the agent through the **Studio UI** — linked from the deployment's page in LangSmith, or available standalone at `https://smith.langchain.com/studio` against any registered graph. Studio is a chat-style interface that streams responses, surfaces HITL interrupts inline, and persists each conversation as a thread that lands in the trace project.

Running a representative batch of queries here validates that the deployed agent behaves the same way the local one did. The queries also feed additional trace volume into your LangSmith project, useful for Module 3's failure-mode discovery features.

<img src="../images/Studio_testing.png" alt="LangSmith Studio mid-conversation with the deployed client_research agent" style="width: auto; max-height: 380px; border-radius: 8px;">

### Test query checklist

Open Studio against the `client_research` graph and run each query in order. Each one targets a distinct trace shape; together they exercise every code path the agent has.

**Profile lookup only (single tool, fastest)**

1. `What does Smith Family Office hold?`
2. `Pull up Acme Pension Fund and tell me about their risk profile.`
3. `What does TechCorp Treasury hold?`

**Research only (research subagent, no client lookup)**

4. `What's the latest news on Apple (AAPL)? Search once.`
5. `Summarize Microsoft's most recent quarterly earnings.`
6. `Research recent news on the energy sector. Search at most once.`

**Multi-step (profile + research, longer trajectories)**

7. `Prep me for my meeting with Smith Family Office. Cover their top three holdings briefly.`
8. `Look up Acme Pension Fund, then research one major holding.`
9. `Write a portfolio update for TechCorp Treasury and save it to /tcorp_update.md.`

**Edge cases (likely to surface failure modes)**

10. `Look up the client 'globex-corp' and tell me about their holdings.`
11. `What was Smith Family Office's exact return last quarter?`
12. `How will AAPL stock perform next month?`


### What to watch for

While running through the checklist:

- **Trajectory** — profile-only queries should show one `get_client_profile` call in the trace tree. Research-only queries should show one `task` (subagent delegation) and one or more `web_search` calls inside the subagent. Multi-step queries should chain `get_client_profile` → `task` → `write_file`.
- **Unknown client handling** — query 10 should report the client is not in the system and list known IDs, not fabricate data.
- **Speculative refusal** — query 12 asks for a prediction the agent can't responsibly make. The compliance rules in the agent's middleware (Module 1, §1.5) should result in a hedged or declined response, not a confident forecast.

Optional: repeat the checklist a second time to roughly double the trace volume. Module 7's Insights Agent benefits from a larger corpus, though the warm-up and trigger prompts from earlier modules will have already produced enough traces for it to work with.


## Recap

| Pillar | What | How |
|---|---|---|
| **Govern** | Workspace-level PII / secrets redaction | `POST /v1/platform/gateway-policies` (`action=block`) |
| **Govern** | Route model calls through the gateway | `init_chat_model(..., base_url="https://gateway.smith.langchain.com/anthropic")` in `utils/models.py` |
| **Deploy** | Deployable graph config | `langgraph.json` at repo root |
| **Deploy** | Agent identity + skills | `AGENTS.md`, `skills/` |
| **Deploy** | Ship it | `langgraph deploy --name langsmith-poc-client-research` |

**The production pattern:** one policy + one `base_url` change + one deploy command. The agent now lives behind a managed server, every model call passes through a governed gateway, and 30+ endpoints are available to call it.

**Next:** Return to `03_finding_failure_modes.ipynb` to analyze the expanded trace corpus from Studio testing using Chat, Insights, and Engine.